# Week 3 Dataset Audit — 高空紅外線無人機影像車輛偵測

學號：7114029015　姓名：黃柏瑜

**資料狀態聲明**
- 資料集 **HIT-UAV**（Suo et al., *Scientific Data* 10, 227, 2023；GitHub repo LICENSE：CC BY 4.0）作為 Dataset v0.2 的 pilot。
- 本 Notebook 只使用官方 `normal_json` 標註檔（影像本體未下載），因此 pixel-level 檢查（影像近重複、亮度、壓縮失真）列為 TODO。

對應 Spec：`evidence/week03_dataset_spec.md`（Dataset Spec v0.2）

## 0. 載入資料（若沒有標註檔則自動下載）

In [1]:
from pathlib import Path
import json, urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "hit_uav" / "normal_json"
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/suojiashun/HIT-UAV-Infrared-Thermal-Dataset/main/normal_json/annotations/"

for split in ["train", "val", "test"]:
    f = DATA_DIR / f"{split}.json"
    if not f.exists():
        urllib.request.urlretrieve(BASE_URL + f"{split}.json", f)
    print(f"{split}.json  {f.stat().st_size:,} bytes")

train.json  2,846,250 bytes
val.json  401,578 bytes
test.json  785,798 bytes


In [2]:
images, anns, cats = [], [], {}
for split in ["train", "val", "test"]:
    j = json.load(open(DATA_DIR / f"{split}.json", encoding="utf-8"))
    cats.update({c["id"]: c["name"] for c in j["categories"]})
    for i in j["images"]:
        i["official_split"] = split; images.append(i)
    for a in j["annotation"]:          # 注意：官方 key 是 "annotation"（單數）
        a["official_split"] = split; anns.append(a)

img = pd.DataFrame(images)
ann = pd.DataFrame(anns)

# 檔名格式 T_H_A_W_N：T=0 白天/1 夜間、H=飛行高度(m)、A=相機俯角(deg)、W=天候碼、N=取樣序號
parts = img["filename"].str.replace(".jpg", "", regex=False).str.split("_", expand=True)
img["daynight"]   = parts[0].astype(int).map({0: "day", 1: "night"})
img["altitude_m"] = parts[1].astype(int)
img["angle_deg"]  = parts[2].astype(int)
img["weather"]    = parts[3]
img["serial"]     = parts[4].astype(int)

ann["category"] = ann["category_id"].map(cats)
ann[["x", "y", "w", "h"]] = pd.DataFrame(ann["bbox"].tolist(), index=ann.index)

print("Categories:", cats)
img.head()

Categories: {0: 'Person', 1: 'Car', 2: 'Bicycle', 3: 'OtherVehicle', 4: 'DontCare'}


,filename,height,width,id,date_captured,official_split,daynight,altitude_m,angle_deg,weather,serial
0,1_130_90_0_04077.jpg,512,640,0,20210116,train,night,130,90,0,4077
1,0_110_50_0_08412.jpg,512,640,1,20210121,train,day,110,50,0,8412
2,1_60_40_0_04117.jpg,512,640,2,20210116,train,night,60,40,0,4117
3,0_80_80_0_02266.jpg,512,640,3,20210115,train,day,80,80,0,2266
4,1_130_50_0_03905.jpg,512,640,4,20210116,train,night,130,50,0,3905


## 1. Shape / Columns / Unit 對齊
W2 Unit = 單張高空紅外線影格。HIT-UAV 一張 image = 一個 640×512 影格 → 一個 Unit；一筆 annotation = 影格內一個物件框。

In [3]:
print("Images (Units):", len(img))
print("Annotations   :", len(ann))
print("Image columns :", list(img.columns))
print("Ann columns   :", list(ann.columns))
print("Resolution    :", img.groupby(["width", "height"]).size().to_dict())
print("Official split (images):", img["official_split"].value_counts().to_dict())

Images (Units): 2898
Annotations   : 24899
Image columns : ['filename', 'height', 'width', 'id', 'date_captured', 'official_split', 'daynight', 'altitude_m', 'angle_deg', 'weather', 'serial']
Ann columns   : ['segmentation', 'area', 'image_id', 'bbox', 'category_id', 'id', 'official_split', 'category', 'x', 'y', 'w', 'h']
Resolution    : {(640, 512): 2898}
Official split (images): {'train': 2029, 'test': 579, 'val': 290}


## 2. Missing Values

In [4]:
print(img.isna().sum()[lambda s: s > 0].to_dict() or "images: no missing values")
print(ann[["image_id", "bbox", "category_id", "area"]].isna().sum().to_dict())
no_ann = (~img["id"].isin(ann["image_id"])).sum()
print("Images with zero annotations:", no_ann)

images: no missing values
{'image_id': 0, 'bbox': 0, 'category_id': 0, 'area': 0}
Images with zero annotations: 32


## 3. Duplicates（exact）

In [5]:
print("Duplicate filename      :", img["filename"].duplicated().sum())
print("Duplicate image id      :", img["id"].duplicated().sum())
print("Duplicate serial number :", img["serial"].duplicated().sum())
print("Duplicate boxes (same image + same bbox):", ann.duplicated(subset=["image_id", "x", "y", "w", "h"]).sum())
print("Duplicate boxes (same image + bbox + class):", ann.duplicated(subset=["image_id", "x", "y", "w", "h", "category_id"]).sum())

Duplicate filename      : 0
Duplicate image id      : 0
Duplicate serial number : 0
Duplicate boxes (same image + same bbox): 0
Duplicate boxes (same image + bbox + class): 0


## 4. Label Distribution 與 Target 對應
本研究 Target = `vehicle`（機動車輛）。v0.2 映射規則：`Car` + `OtherVehicle` → `vehicle`；`Bicycle` 不屬於 Target；`DontCare` 作為評估忽略區；`Person` 不屬於本研究 Target。

In [6]:
VEHICLE = ["Car", "OtherVehicle"]
label_counts = ann["category"].value_counts()
print(pd.DataFrame({"count": label_counts, "pct": (label_counts / len(ann) * 100).round(1)}))

veh = ann[ann["category"].isin(VEHICLE)].merge(
    img[["id", "filename", "date_captured", "daynight", "altitude_m", "angle_deg", "serial", "official_split"]],
    left_on="image_id", right_on="id", suffixes=("", "_img"))
img["n_vehicle"] = img["id"].map(veh.groupby("image_id").size()).fillna(0).astype(int)

print("\nVehicle boxes:", len(veh), veh["category"].value_counts().to_dict())
print("OtherVehicle share of vehicle boxes: %.1f%%" % (100 * (veh["category"] == "OtherVehicle").mean()))
print("Images with >=1 vehicle:", (img["n_vehicle"] > 0).sum(), f"({(img['n_vehicle'] > 0).mean()*100:.1f}%)")
print("Images with 0 vehicle  :", (img["n_vehicle"] == 0).sum())
print("Vehicles per image (images with vehicle):", img.loc[img["n_vehicle"] > 0, "n_vehicle"].describe().round(2).to_dict())

              count   pct
category                 
Person        12312  49.4
Car            7311  29.4
Bicycle        4980  20.0
DontCare        148   0.6
OtherVehicle    148   0.6

Vehicle boxes: 7459 {'Car': 7311, 'OtherVehicle': 148}
OtherVehicle share of vehicle boxes: 2.0%
Images with >=1 vehicle: 1424 (49.1%)
Images with 0 vehicle  : 1474
Vehicles per image (images with vehicle): {'count': 1424.0, 'mean': 5.24, 'std': 4.71, 'min': 1.0, '25%': 2.0, '50%': 4.0, '75%': 7.0, 'max': 32.0}


## 5. Bounding Box 有效性（Invalid / Outlier）

In [7]:
oob = ((ann["x"] < 0) | (ann["y"] < 0) | (ann["x"] + ann["w"] > 640) | (ann["y"] + ann["h"] > 512)).sum()
degenerate = ((veh["w"] <= 2) | (veh["h"] <= 2)).sum()
area_mismatch = (ann["area"] != ann["w"] * ann["h"]).sum()
print("Out-of-bound boxes (all classes):", oob)
print("Degenerate vehicle boxes (w or h <= 2px):", degenerate)
print("area != w*h:", area_mismatch)
veh["side_px"] = np.sqrt(veh["w"] * veh["h"])
veh["aspect"] = veh[["w", "h"]].max(axis=1) / veh[["w", "h"]].min(axis=1)
print("Vehicle box side (sqrt(area)) px:", veh["side_px"].describe().round(1).to_dict())
print("Aspect ratio > 4:", (veh["aspect"] > 4).sum())

Out-of-bound boxes (all classes): 0
Degenerate vehicle boxes (w or h <= 2px): 0
area != w*h: 0
Vehicle box side (sqrt(area)) px: {'count': 7459.0, 'mean': 51.4, 'std': 25.1, 'min': 7.2, '25%': 32.3, '50%': 46.8, '75%': 66.5, 'max': 200.7}
Aspect ratio > 4: 18


## 6. 目標尺度 vs W2 假設（微小目標 < 16×16）

In [8]:
bins = [0, 16**2, 32**2, np.inf]
labels = ["tiny (<16x16)", "small (16x16~32x32)", "medium+ (>=32x32)"]
veh["scale"] = pd.cut(veh["w"] * veh["h"], bins=bins, labels=labels, right=False)
sc = veh["scale"].value_counts().reindex(labels)
print(pd.DataFrame({"count": sc, "pct": (sc / len(veh) * 100).round(1)}))
print("\nMedian vehicle side (px) by altitude:")
print(veh.groupby("altitude_m")["side_px"].median().round(1).to_string())
print("\nTiny share by altitude (%):")
print((veh.assign(t=veh["scale"] == labels[0]).groupby("altitude_m")["t"].mean() * 100).round(1).to_string())

                     count   pct
scale                           
tiny (<16x16)          184   2.5
small (16x16~32x32)   1645  22.1
medium+ (>=32x32)     5630  75.5

Median vehicle side (px) by altitude:
altitude_m
60     76.9
70     60.3
80     45.3
90     42.0
100    50.2
110    45.9
120    43.5
130    29.6

Tiny share by altitude (%):
altitude_m
60     0.0
70     0.1
80     0.6
90     1.2
100    3.1
110    0.0
120    0.5
130    9.5


## 7. Time Range / Coverage

In [9]:
print("Date range:", img["date_captured"].min(), "→", img["date_captured"].max(), "| dates:", img["date_captured"].nunique())
print("Weather codes:", img["weather"].value_counts().to_dict())
print("\nImages by day/night:", img["daynight"].value_counts().to_dict())
print("Vehicle boxes by day/night:", veh["daynight"].value_counts().to_dict())
print("\nImages by date x day/night")
print(pd.crosstab(img["date_captured"], img["daynight"]))
print("\nVehicle boxes by date x day/night")
print(pd.crosstab(veh["date_captured"], veh["daynight"]))
print("\nImages by altitude:", img["altitude_m"].value_counts().sort_index().to_dict())
print("Images by angle   :", img["angle_deg"].value_counts().sort_index().to_dict())

Date range: 20201217 → 20210123 | dates: 9
Weather codes: {'0': 2898}

Images by day/night: {'night': 1981, 'day': 917}
Vehicle boxes by day/night: {'night': 4914, 'day': 2545}

Images by date x day/night
daynight       day  night
date_captured            
20201217         0    587
20210114         0    162
20210115       191    148
20210116       111    204
20210118         0    125
20210119       148    168
20210120       164    262
20210121       153    210
20210123       150    115

Vehicle boxes by date x day/night
daynight       day  night
date_captured            
20210114         0    134
20210115        10    185
20210116        68   1189
20210118         0    264
20210119       496    767
20210120       614   1243
20210121       950    775
20210123       407    357

Images by altitude: {60: 863, 70: 267, 80: 256, 90: 145, 100: 291, 110: 247, 120: 336, 130: 493}
Images by angle   : {30: 477, 40: 339, 50: 486, 60: 471, 70: 421, 80: 390, 90: 314}


## 8. Group IDs 與官方 Split 的 Leakage 檢查
HIT-UAV 沒有 `video_id`。每支飛行影片記錄了日期、日夜、高度、角度，因此以 `flight_group = date + daynight + altitude + angle` 作為 **影片／飛行段 proxy**。`serial` 是取樣序號（官方論文：7 FPS 影片每 15 frames 取 1 張），serial 差 1 推估約相隔 2 秒。

In [10]:
img["flight_group"] = img["date_captured"] + "_" + img["daynight"] + "_" + img["altitude_m"].astype(str) + "_" + img["angle_deg"].astype(str)
g = img.groupby("flight_group")["serial"].agg(["min", "max", "count"])
print("flight_group 數量:", len(g))
print("每組影像數:", g["count"].describe().round(1).to_dict())
# serial 區間是否不重疊（支持 flight_group ≈ 一段連續影片）
gs = g.sort_values("min")
print("serial 區間互相重疊的相鄰組數:", int((gs["min"].values[1:] <= gs["max"].values[:-1]).sum()))

span = img.groupby("flight_group")["official_split"].nunique()
print("\n跨越 >1 個官方 split 的 flight_group:", int((span > 1).sum()), "/", len(span))

def neighbor_gap(test_df, train_df):
    tr = train_df.groupby("flight_group")["serial"].apply(lambda s: np.sort(s.values)).to_dict()
    out = []
    for fg, s in zip(test_df["flight_group"], test_df["serial"]):
        out.append(np.min(np.abs(tr[fg] - s)) if fg in tr else np.inf)
    return np.array(out)

off_test = img[img["official_split"] == "test"]
off_train = img[img["official_split"] == "train"]
gap = neighbor_gap(off_test, off_train)
off_same_group = np.isfinite(gap).mean() * 100
off_adjacent = (gap <= 1).mean() * 100
print(f"官方 Test 影像中，Train 有同一 flight_group 的比例: {off_same_group:.1f}%")
print(f"官方 Test 影像中，Train 有相鄰取樣 (serial 差 1, ≈2 秒) 的比例: {off_adjacent:.1f}%")
print(f"serial 差 <= 2 的比例: {(gap <= 2).mean()*100:.1f}%")

flight_group 數量: 264
每組影像數: {'count': 264.0, 'mean': 11.0, 'std': 15.9, 'min': 2.0, '25%': 6.0, '50%': 8.0, '75%': 10.2, 'max': 150.0}
serial 區間互相重疊的相鄰組數: 0

跨越 >1 個官方 split 的 flight_group: 240 / 264
官方 Test 影像中，Train 有同一 flight_group 的比例: 100.0%
官方 Test 影像中，Train 有相鄰取樣 (serial 差 1, ≈2 秒) 的比例: 31.3%
serial 差 <= 2 的比例: 49.4%


## 9. v0.2 Split：依拍攝日期做 Group／時間切分
Test 要模擬「**未見過的新飛行任務日**」。規則：`date_captured` 最後兩天（2021-01-21、2021-01-23）→ Test；2021-01-19 → Val；其餘日期 → Train。三個集合都需涵蓋白天與夜間。

In [11]:
TEST_DATES = ["20210121", "20210123"]
VAL_DATES = ["20210119"]
img["split_v02"] = np.where(img["date_captured"].isin(TEST_DATES), "test",
                   np.where(img["date_captured"].isin(VAL_DATES), "val", "train"))
veh["split_v02"] = veh["image_id"].map(img.set_index("id")["split_v02"])

summary = pd.DataFrame({
    "images": img.groupby("split_v02").size(),
    "images_with_vehicle": img[img["n_vehicle"] > 0].groupby("split_v02").size(),
    "vehicle_boxes": veh.groupby("split_v02").size(),
    "other_vehicle": veh[veh["category"] == "OtherVehicle"].groupby("split_v02").size(),
    "dates": img.groupby("split_v02")["date_captured"].nunique(),
}).reindex(["train", "val", "test"])
summary["img_pct"] = (summary["images"] / len(img) * 100).round(1)
print(summary)

print("\nDay/Night 影像數 by split")
print(pd.crosstab(img["split_v02"], img["daynight"]).reindex(["train", "val", "test"]))
print("\nAltitude 覆蓋 by split")
print(pd.crosstab(img["split_v02"], img["altitude_m"]).reindex(["train", "val", "test"]))

tr_groups = set(img.loc[img["split_v02"] == "train", "flight_group"])
te_groups = set(img.loc[img["split_v02"] == "test", "flight_group"])
va_groups = set(img.loc[img["split_v02"] == "val", "flight_group"])
print("\nGroup overlap train∩test:", len(tr_groups & te_groups), "| train∩val:", len(tr_groups & va_groups), "| val∩test:", len(va_groups & te_groups))
date_overlap = set(img.loc[img["split_v02"] == "train", "date_captured"]) & set(img.loc[img["split_v02"] == "test", "date_captured"])
print("Date overlap train∩test:", len(date_overlap))
v02_gap = neighbor_gap(img[img["split_v02"] == "test"], img[img["split_v02"] == "train"])
print(f"v0.2 Test 影像在 Train 有相鄰取樣的比例: {(v02_gap <= 1).mean()*100:.1f}%")

           images  images_with_vehicle  vehicle_boxes  other_vehicle  dates  img_pct
split_v02                                                                           
train        1954                  617           3707             46      6     67.4
val           316                  310           1263             25      1     10.9
test          628                  497           2489             77      2     21.7

Day/Night 影像數 by split
daynight   day  night
split_v02            
train      466   1488
val        148    168
test       303    325

Altitude 覆蓋 by split
altitude_m  60   70   80   90   100  110  120  130
split_v02                                         
train       863  267  198   96  146   92   54  238
val           0    0    0    0   54   49  144   69
test          0    0   58   49   91  106  138  186

Group overlap train∩test: 0 | train∩val: 0 | val∩test: 0
Date overlap train∩test: 0
v0.2 Test 影像在 Train 有相鄰取樣的比例: 0.0%


## 10. 欄位 Leakage 檢查（哪些欄位不能當 Input）

In [12]:
field_roles = pd.DataFrame([
    ("image pixels (640x512 IR)", "Input", "是", "推論當下唯一允許進入模型的資料"),
    ("altitude_m / angle_deg (檔名)", "後處理 only", "是（飛控即時可得）", "只能做尺寸過濾；不可進模型，也不可用來挑 Test 門檻"),
    ("bbox / category_id", "Target", "否（人工事後標註）", "Label"),
    ("DontCare 區域", "評估忽略區", "否", "標註者無法判定的物件，不算 FP/FN"),
    ("date_captured", "Split key", "否（離線 metadata）", "用於切分；不可當特徵"),
    ("daynight", "分組評估", "可推得", "只用於分組報告 Recall，不進模型"),
    ("serial / filename / image id", "ID", "否", "編碼了拍攝順序與條件，不可當特徵"),
    ("official_split", "禁用", "否", "官方切分有跨組洩漏，v0.2 不採用"),
], columns=["field", "role", "decision-time available", "note"])
field_roles

,field,role,decision-time available,note
0,image pixels (640x512 IR),Input,是,推論當下唯一允許進入模型的資料
1,altitude_m / angle_deg (檔名),後處理 only,是（飛控即時可得）,只能做尺寸過濾；不可進模型，也不可用來挑 Test 門檻
2,bbox / category_id,Target,否（人工事後標註）,Label
3,DontCare 區域,評估忽略區,否,標註者無法判定的物件，不算 FP/FN
4,date_captured,Split key,否（離線 metadata）,用於切分；不可當特徵
5,daynight,分組評估,可推得,只用於分組報告 Recall，不進模型
6,serial / filename / image id,ID,否,編碼了拍攝順序與條件，不可當特徵
7,official_split,禁用,否,官方切分有跨組洩漏，v0.2 不採用


## 11. 匯出 Audit Metrics

In [13]:
metrics = pd.DataFrame([
    ("images", len(img)),
    ("annotations_all_classes", len(ann)),
    ("vehicle_boxes (Car+OtherVehicle)", len(veh)),
    ("other_vehicle_pct_of_vehicle", round(100 * (veh["category"] == "OtherVehicle").mean(), 1)),
    ("images_with_vehicle", int((img["n_vehicle"] > 0).sum())),
    ("images_without_any_annotation", int(no_ann)),
    ("missing_values", int(img.isna().sum().sum() + ann[["bbox", "category_id"]].isna().sum().sum())),
    ("duplicate_boxes", int(ann.duplicated(subset=["image_id", "x", "y", "w", "h"]).sum())),
    ("out_of_bound_boxes", int(oob)),
    ("degenerate_vehicle_boxes", int(degenerate)),
    ("tiny_vehicle_pct (<16x16)", round(float(sc.iloc[0] / len(veh) * 100), 1)),
    ("medium_plus_vehicle_pct (>=32x32)", round(float(sc.iloc[2] / len(veh) * 100), 1)),
    ("flight_groups", int(len(g))),
    ("official_test_same_group_in_train_pct", round(off_same_group, 1)),
    ("official_test_adjacent_frame_in_train_pct", round(off_adjacent, 1)),
    ("v02_group_overlap_train_test", len(tr_groups & te_groups)),
], columns=["audit_item", "value"])
out = ROOT / "outputs"; out.mkdir(exist_ok=True)
metrics.to_csv(out / "week03_audit_metrics.csv", index=False, encoding="utf-8-sig")
metrics

,audit_item,value
0,images,2898.0
1,annotations_all_classes,24899.0
2,vehicle_boxes (Car+OtherVehicle),7459.0
3,other_vehicle_pct_of_vehicle,2.0
4,images_with_vehicle,1424.0
5,images_without_any_annotation,32.0
6,missing_values,0.0
7,duplicate_boxes,0.0
8,out_of_bound_boxes,0.0
9,degenerate_vehicle_boxes,0.0


## 12. Audit Conclusion

- 這份資料可以支持 W2 的 Unit（單張高空 IR 影格）與 `vehicle` Target 的 **pilot 實驗**；欄位齊全、無缺值、無重複框、無越界框。
- **不能直接支持 W2「微小目標」主張**：大部分車輛框 ≥ 32×32（見第 6 節），W2 Success Criteria 中的 tiny-object 指標在本資料上樣本很少。
- **官方 split 不可直接使用**：Test 影像幾乎都與 Train 共用同一段飛行影片（見第 8 節），改用依日期的 Group／Time Split（第 9 節，group overlap = 0）。
- 詳細 Finding（Problem → Evidence → Impact → Decision）寫在 `evidence/week03_dataset_spec.md` 的 C 節。